In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import LogisticRegression
from sklearn.metrics import make_scorer, matthews_corrcoef

import sys
sys.path.append("../../utils/")

from utils import *

In [ ]:
# ===== RUTAS =====
PROJECT_ROOT = Path.cwd().resolve().parents[2]

NOMBRE_EXPERIMENTO = "CIC17__NearMiss_SMOTE_ENN__v1__pca4_logreg__v1"
CARPETA_DATASET = "CIC17__NearMiss_SMOTE_ENN__v1"
NOMBRE_DATASET_LIMPIO = f"{CARPETA_DATASET}__train.csv"

RUTA_DATASET = PROJECT_ROOT / "02_datasets" / "processed" / CARPETA_DATASET
RUTA_RESULTADOS = PROJECT_ROOT / "04_experimentos" / "logs" / "resultados" / NOMBRE_EXPERIMENTO

NOMBRE_RESULTADOS_CSV = f"{NOMBRE_EXPERIMENTO}__folds.csv"
NOMBRE_RESULTADOS_JSON = f"{NOMBRE_EXPERIMENTO}__summary.json"

# ===== PARÁMETROS =====
LABEL_COL = "LABEL"

N_SPLITS = 5
SHUFFLE = True
RANDOM_STATE = 42

# ===== CONFIG PCA =====
N_COMPONENTS_PCA = 4

In [ ]:
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

print("Ruta dataset limpio:")
print(RUTA_DATASET.resolve())
print()

print("Ruta resultados:")
print(RUTA_RESULTADOS.resolve())

In [ ]:
df = cargar_dataset(
    nombre_dataset=NOMBRE_DATASET_LIMPIO
    ruta_base=RUTA_DATASET
)

print("Forma del dataset limpio:")
print(df.shape)

df.head()

In [ ]:
if LABEL_COL not in df.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL}")

print("Última columna:", df.columns[-1])
print("Tipo de LABEL:", df[LABEL_COL].dtype)
print()
print("Distribución de clases:")
display(df[LABEL_COL].value_counts(dropna=False).to_frame("count"))

In [ ]:
X = df.drop(columns=[LABEL_COL]).copy()
y = df[LABEL_COL].copy()

print("Shape X:", X.shape)
print("Shape y:", y.shape)

In [ ]:
columnas_no_numericas = X.select_dtypes(exclude=[np.number]).columns.tolist()

print("Columnas no numéricas en X:")
print(columnas_no_numericas)

if len(columnas_no_numericas) > 0:
    raise ValueError("Hay columnas no numéricas en X. Revísalas antes de seguir.")

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=N_COMPONENTS_PCA)),
    ("logreg", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        n_jobs=-1
    ))
])

pipeline

In [ ]:
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=SHUFFLE,
    random_state=RANDOM_STATE
)

cv

In [ ]:
scoring = {
    "accuracy": "accuracy",
    "precision_weighted": "precision_weighted",
    "recall_weighted": "recall_weighted",
    "f1_weighted": "f1_weighted",
    
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro",
    
    "mcc": make_scorer(matthews_corrcoef)
}

In [ ]:
cv_results = cross_validate(
    estimator=pipeline,
    X=X,
    y=y,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=-1
)

cv_results.keys()

In [ ]:
df_folds = pd.DataFrame({
    "fold": np.arange(1, N_SPLITS + 1),

    "accuracy": cv_results["test_accuracy"],

    "precision_weighted": cv_results["test_precision_weighted"],
    "recall_weighted": cv_results["test_recall_weighted"],
    "f1_weighted": cv_results["test_f1_weighted"],

    "precision_macro": cv_results["test_precision_macro"],
    "recall_macro": cv_results["test_recall_macro"],
    "f1_macro": cv_results["test_f1_macro"],

    "mcc": cv_results["test_mcc"],

    "fit_time": cv_results["fit_time"],
    "score_time": cv_results["score_time"]
})

df_folds

In [ ]:
summary = {
    "experimento": NOMBRE_EXPERIMENTO,
    "dataset": str(RUTA_DATASET),
    "shape_dataset": {
        "rows": int(df.shape[0]),
        "cols": int(df.shape[1])
    },
    "parametros": {
        "label_col": LABEL_COL,
        "n_splits": N_SPLITS,
        "shuffle": SHUFFLE,
        "random_state": RANDOM_STATE,
        "n_neighbors": N_NEIGHBORS,
        "weights": WEIGHTS,
        "metric": METRIC,
        "p": P,
        "n_components_pca": N_COMPONENTS_PCA
    },
    "metricas_media": {
        "accuracy": float(df_folds["accuracy"].mean()),

        "precision_weighted": float(df_folds["precision_weighted"].mean()),
        "recall_weighted": float(df_folds["recall_weighted"].mean()),
        "f1_weighted": float(df_folds["f1_weighted"].mean()),

        "precision_macro": float(df_folds["precision_macro"].mean()),
        "recall_macro": float(df_folds["recall_macro"].mean()),
        "f1_macro": float(df_folds["f1_macro"].mean()),

        "mcc": float(df_folds["mcc"].mean()),
        "fit_time": float(df_folds["fit_time"].mean()),
        "score_time": float(df_folds["score_time"].mean())
    },
    "metricas_std": {
        "accuracy": float(df_folds["accuracy"].std(ddof=1)),

        "precision_weighted": float(df_folds["precision_weighted"].std(ddof=1)),
        "recall_weighted": float(df_folds["recall_weighted"].std(ddof=1)),
        "f1_weighted": float(df_folds["f1_weighted"].std(ddof=1)),

        "precision_macro": float(df_folds["precision_macro"].std(ddof=1)),
        "recall_macro": float(df_folds["recall_macro"].std(ddof=1)),
        "f1_macro": float(df_folds["f1_macro"].std(ddof=1)),

        "mcc": float(df_folds["mcc"].std(ddof=1)),
        "fit_time": float(df_folds["fit_time"].std(ddof=1)),
        "score_time": float(df_folds["score_time"].std(ddof=1))
    }
}

summary

In [ ]:
print("========== RESULTADOS CV ==========")
print(f"Accuracy            : {summary['metricas_media']['accuracy']:.6f} ± {summary['metricas_std']['accuracy']:.6f}")

print(f"Precision weighted  : {summary['metricas_media']['precision_weighted']:.6f} ± {summary['metricas_std']['precision_weighted']:.6f}")
print(f"Recall weighted     : {summary['metricas_media']['recall_weighted']:.6f} ± {summary['metricas_std']['recall_weighted']:.6f}")
print(f"F1 weighted         : {summary['metricas_media']['f1_weighted']:.6f} ± {summary['metricas_std']['f1_weighted']:.6f}")

print(f"Precision macro     : {summary['metricas_media']['precision_macro']:.6f} ± {summary['metricas_std']['precision_macro']:.6f}")
print(f"Recall macro        : {summary['metricas_media']['recall_macro']:.6f} ± {summary['metricas_std']['recall_macro']:.6f}")
print(f"F1 macro            : {summary['metricas_media']['f1_macro']:.6f} ± {summary['metricas_std']['f1_macro']:.6f}")

print(f"MCC                 : {summary['metricas_media']['mcc']:.6f} ± {summary['metricas_std']['mcc']:.6f}")
print()
print(f"Fit time medio      : {summary['metricas_media']['fit_time']:.6f}")
print(f"Score time medio    : {summary['metricas_media']['score_time']:.6f}")

In [ ]:
ruta_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_CSV
df_folds.to_csv(ruta_csv, index=False)

print("Resultados por fold guardados en:")
print(ruta_csv.resolve())

In [ ]:
ruta_json = RUTA_RESULTADOS / NOMBRE_RESULTADOS_JSON

with open(ruta_json, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4, ensure_ascii=False)

print("Resumen JSON guardado en:")
print(ruta_json.resolve())

In [ ]:
df_folds